# SongUNet with Covariance-Weighted Tikhonov — Rescaled Matern, $N=2$

**Purpose.** Test whether covariance-weighted Tikhonov regularization preserves coarse-band
content while regularizing fine bands in a **trained SongUNet** (the NVlabs/edm architecture
used by Baptista et al.), on the rescaled Matern geometry at $n_{\text{train}}=2$.

The closed-form GMM study (`gmm_tikhonov_variants_comparison.ipynb`) showed covariance-weighted
Tikhonov is *scale-selective* — it de-memorizes the fine band at $c$ ~100x smaller than plain
Tikhonov while leaving the coarse band memorized. The earlier `edm_unet_covariance_tikhonov.ipynb`
showed the SmallUNet tracks the GMM at $n_{\text{train}}=8$. This notebook tests the same
claim with the full SongUNet+EDMPrecond pipeline from Baptista et al., at $n_{\text{train}}=2$
with the rescaled geometry that `baptista_config_matern_n2.ipynb` established.

**Arms (25 total):**
1. **Gate** ($c=0$): unregularized. Must reproduce `arm_rescaled_c16_result.pt`.
2. **Isotropic Tikhonov** (4 values of $c$): $\Gamma = (c/\sigma^2) I$.
3. **Covariance-empirical** (4 values of $c$ x 4 pool sizes $n \in \{2,8,16,32\}$): $\Gamma = (c/\sigma^2) \hat\Sigma^{-1}_n$.
4. **Covariance-population** (4 values of $c$): $\Gamma = (c/\sigma^2) \Sigma^{-1}$.

All arms use `model_channels=16` (880,097 params), the cheapest capacity that fully memorizes
at $n=2$ per the source notebook.

**Metric convention.** `exclude_nn=True`, `aggregate='mean_of_ratios'` (the default),
bands: coarse (0.5,4), mid1 (4,10), mid2 (10,18), fine (18,32), `sigma_max=80`.

---

**Cluster execution:**
```bash
caffeinate -i python3 -m nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 \
  notebooks/multiscale/songunet_covariance_tikhonov.ipynb
```

**Note on `sigma_data`.** The default is 0.5 (NVlabs/edm default, hardcoded in Baptista's code).
The rescaled data has std ~ 0.10. A `sigma_data=0.1` arm should be run as a cheap check, but
`sigma_data=0.5` is the primary for consistency with the source notebook.

## Setup

In [ ]:
import os, sys, math, time, copy, json
import numpy as np
import torch
import matplotlib

# Headless-safe: picks Agg under SLURM (no $DISPLAY), leaves inline alone in Jupyter.
if not os.environ.get('DISPLAY') and not hasattr(sys, 'ps1'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

# -- path setup --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from device_utils import resolve_device

DEVICE = resolve_device()           # cuda > mps > cpu
torch.backends.cudnn.benchmark = True

N_TRAIN = 2      # main.py:13
GRID    = 128    # our fields (main.py's rectangles are 64)

# Redirect heavy artifacts off a shared quota:  export FIELD_RESULTS_DIR=$SCRATCH/...
results_dir = os.environ.get('FIELD_RESULTS_DIR',
                             os.path.join(repo_root, 'results', 'data'))
fig_dir = os.environ.get('FIELD_FIG_DIR',
                         os.path.join(repo_root, 'results', 'figures'))
os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

print(f'torch {torch.__version__}')
print(f'results -> {results_dir}')
print(f'figures -> {fig_dir}')
print(f'device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name}, {p.total_memory / 1e9:.1f} GB, cc {p.major}.{p.minor}')
else:
    print('WARNING: this notebook is sized for a CUDA GPU. 50k epochs at 128x128 is not '
          'practical on CPU/MPS -- raise SMOKE if you are just checking the pipeline runs.')

## The EDM network -- verbatim

Copied byte-for-byte from `baptista_config_matern_n2.ipynb`, which transcribes
`RectangleImages/training/networks.py` (byte-identical to
[NVlabs/edm](https://github.com/NVlabs/edm) apart from comments) minus the `@persistence`
decorators and the unused `DhariwalUNet` / `VPPrecond` / `VEPrecond` / `iDDPMPrecond`.

> Karras, Aittala, Aila & Laine, *Elucidating the Design Space of Diffusion-Based Generative
> Models*, NeurIPS 2022. Code (c) 2022 NVIDIA CORPORATION & AFFILIATES, released under
> CC BY-NC-SA 4.0.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Transcribed verbatim from RectangleImages/training/networks.py in baptistar/DiffusionModelDynamics
# (byte-identical to NVlabs/edm training/networks.py apart from comments).
#
# Copyright (c) 2022, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# Licensed under CC BY-NC-SA 4.0 -- http://creativecommons.org/licenses/by-nc-sa/4.0/
# "Elucidating the Design Space of Diffusion-Based Generative Models", Karras et al., NeurIPS 2022.
#
# Changes: @persistence decorators and the torch_utils import removed (pickle plumbing only);
# DhariwalUNet / VPPrecond / VEPrecond / iDDPMPrecond omitted (unused by main.py).
# No computational change.
# ---------------------------------------------------------------------------------------------

from torch.nn.functional import silu

def weight_init(shape, mode, fan_in, fan_out):
    if mode == 'xavier_uniform': return np.sqrt(6 / (fan_in + fan_out)) * (torch.rand(*shape) * 2 - 1)
    if mode == 'xavier_normal':  return np.sqrt(2 / (fan_in + fan_out)) * torch.randn(*shape)
    if mode == 'kaiming_uniform': return np.sqrt(3 / fan_in) * (torch.rand(*shape) * 2 - 1)
    if mode == 'kaiming_normal':  return np.sqrt(1 / fan_in) * torch.randn(*shape)
    raise ValueError(f'Invalid init mode "{mode}"')

#----------------------------------------------------------------------------
# Fully-connected layer.

class Linear(torch.nn.Module):
    def __init__(self, in_features, out_features, bias=True, init_mode='kaiming_normal', init_weight=1, init_bias=0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        init_kwargs = dict(mode=init_mode, fan_in=in_features, fan_out=out_features)
        self.weight = torch.nn.Parameter(weight_init([out_features, in_features], **init_kwargs) * init_weight)
        self.bias = torch.nn.Parameter(weight_init([out_features], **init_kwargs) * init_bias) if bias else None

    def forward(self, x):
        x = x @ self.weight.to(x.dtype).t()
        if self.bias is not None:
            x = x.add_(self.bias.to(x.dtype))
        return x

#----------------------------------------------------------------------------
# Convolutional layer with optional up/downsampling.

class Conv2d(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, kernel, bias=True, up=False, down=False,
        resample_filter=[1,1], fused_resample=False, init_mode='kaiming_normal', init_weight=1, init_bias=0,
    ):
        assert not (up and down)
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.up = up
        self.down = down
        self.fused_resample = fused_resample
        init_kwargs = dict(mode=init_mode, fan_in=in_channels*kernel*kernel, fan_out=out_channels*kernel*kernel)
        self.weight = torch.nn.Parameter(weight_init([out_channels, in_channels, kernel, kernel], **init_kwargs) * init_weight) if kernel else None
        self.bias = torch.nn.Parameter(weight_init([out_channels], **init_kwargs) * init_bias) if kernel and bias else None
        f = torch.as_tensor(resample_filter, dtype=torch.float32)
        f = f.ger(f).unsqueeze(0).unsqueeze(1) / f.sum().square()
        self.register_buffer('resample_filter', f if up or down else None)

    def forward(self, x):
        w = self.weight.to(x.dtype) if self.weight is not None else None
        b = self.bias.to(x.dtype) if self.bias is not None else None
        f = self.resample_filter.to(x.dtype) if self.resample_filter is not None else None
        w_pad = w.shape[-1] // 2 if w is not None else 0
        f_pad = (f.shape[-1] - 1) // 2 if f is not None else 0

        if self.fused_resample and self.up and w is not None:
            x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=max(f_pad - w_pad, 0))
            x = torch.nn.functional.conv2d(x, w, padding=max(w_pad - f_pad, 0))
        elif self.fused_resample and self.down and w is not None:
            x = torch.nn.functional.conv2d(x, w, padding=w_pad+f_pad)
            x = torch.nn.functional.conv2d(x, f.tile([self.out_channels, 1, 1, 1]), groups=self.out_channels, stride=2)
        else:
            if self.up:
                x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if self.down:
                x = torch.nn.functional.conv2d(x, f.tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if w is not None:
                x = torch.nn.functional.conv2d(x, w, padding=w_pad)
        if b is not None:
            x = x.add_(b.reshape(1, -1, 1, 1))
        return x

#----------------------------------------------------------------------------
# Group normalization.

class GroupNorm(torch.nn.Module):
    def __init__(self, num_channels, num_groups=32, min_channels_per_group=4, eps=1e-5):
        super().__init__()
        self.num_groups = min(num_groups, num_channels // min_channels_per_group)
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones(num_channels))
        self.bias = torch.nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        x = torch.nn.functional.group_norm(x, num_groups=self.num_groups, weight=self.weight.to(x.dtype), bias=self.bias.to(x.dtype), eps=self.eps)
        return x

#----------------------------------------------------------------------------
# Attention weight computation, i.e., softmax(Q^T * K).
# Performs all computation using FP32, but uses the original datatype for
# inputs/outputs/gradients to conserve memory.

class AttentionOp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k):
        w = torch.einsum('ncq,nck->nqk', q.to(torch.float32), (k / np.sqrt(k.shape[1])).to(torch.float32)).softmax(dim=2).to(q.dtype)
        ctx.save_for_backward(q, k, w)
        return w

    @staticmethod
    def backward(ctx, dw):
        q, k, w = ctx.saved_tensors
        db = torch._softmax_backward_data(grad_output=dw.to(torch.float32), output=w.to(torch.float32), dim=2, input_dtype=torch.float32)
        dq = torch.einsum('nck,nqk->ncq', k.to(torch.float32), db).to(q.dtype) / np.sqrt(k.shape[1])
        dk = torch.einsum('ncq,nqk->nck', q.to(torch.float32), db).to(k.dtype) / np.sqrt(k.shape[1])
        return dq, dk

#----------------------------------------------------------------------------
# Unified U-Net block with optional up/downsampling and self-attention.
# Represents the union of all features employed by the DDPM++, NCSN++, and
# ADM architectures.

class UNetBlock(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, emb_channels, up=False, down=False, attention=False,
        num_heads=None, channels_per_head=64, dropout=0, skip_scale=1, eps=1e-5,
        resample_filter=[1,1], resample_proj=False, adaptive_scale=True,
        init=dict(), init_zero=dict(init_weight=0), init_attn=None,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.emb_channels = emb_channels
        self.num_heads = 0 if not attention else num_heads if num_heads is not None else out_channels // channels_per_head
        self.dropout = dropout
        self.skip_scale = skip_scale
        self.adaptive_scale = adaptive_scale

        self.norm0 = GroupNorm(num_channels=in_channels, eps=eps)
        self.conv0 = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=3, up=up, down=down, resample_filter=resample_filter, **init)
        self.affine = Linear(in_features=emb_channels, out_features=out_channels*(2 if adaptive_scale else 1), **init)
        self.norm1 = GroupNorm(num_channels=out_channels, eps=eps)
        self.conv1 = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=3, **init_zero)

        self.skip = None
        if out_channels != in_channels or up or down:
            kernel = 1 if resample_proj or out_channels!= in_channels else 0
            self.skip = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=kernel, up=up, down=down, resample_filter=resample_filter, **init)

        if self.num_heads:
            self.norm2 = GroupNorm(num_channels=out_channels, eps=eps)
            self.qkv = Conv2d(in_channels=out_channels, out_channels=out_channels*3, kernel=1, **(init_attn if init_attn is not None else init))
            self.proj = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=1, **init_zero)

    def forward(self, x, emb):
        orig = x
        x = self.conv0(silu(self.norm0(x)))

        params = self.affine(emb).unsqueeze(2).unsqueeze(3).to(x.dtype)
        if self.adaptive_scale:
            scale, shift = params.chunk(chunks=2, dim=1)
            x = silu(torch.addcmul(shift, self.norm1(x), scale + 1))
        else:
            x = silu(self.norm1(x.add_(params)))

        x = self.conv1(torch.nn.functional.dropout(x, p=self.dropout, training=self.training))
        x = x.add_(self.skip(orig) if self.skip is not None else orig)
        x = x * self.skip_scale

        if self.num_heads:
            q, k, v = self.qkv(self.norm2(x)).reshape(x.shape[0] * self.num_heads, x.shape[1] // self.num_heads, 3, -1).unbind(2)
            w = AttentionOp.apply(q, k)
            a = torch.einsum('nqk,nck->ncq', w, v)
            x = self.proj(a.reshape(*x.shape)).add_(x)
            x = x * self.skip_scale
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the DDPM++ and ADM architectures.

class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=10000, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        freqs = torch.arange(start=0, end=self.num_channels//2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the NCSN++ architecture.

class FourierEmbedding(torch.nn.Module):
    def __init__(self, num_channels, scale=16):
        super().__init__()
        self.register_buffer('freqs', torch.randn(num_channels // 2) * scale)

    def forward(self, x):
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Reimplementation of the DDPM++ and NCSN++ architectures from the paper
# "Score-Based Generative Modeling through Stochastic Differential
# Equations". Equivalent to the original implementation by Song et al.,
# available at https://github.com/yang-song/score_sde_pytorch

class SongUNet(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution at input/output.
        in_channels,                        # Number of color channels at input.
        out_channels,                       # Number of color channels at output.
        label_dim           = 0,            # Number of class labels, 0 = unconditional.
        augment_dim         = 0,            # Augmentation label dimensionality, 0 = no augmentation.

        model_channels      = 128,          # Base multiplier for the number of channels.
        channel_mult        = [1,2,2,2],    # Per-resolution multipliers for the number of channels.
        channel_mult_emb    = 4,            # Multiplier for the dimensionality of the embedding vector.
        num_blocks          = 4,            # Number of residual blocks per resolution.
        attn_resolutions    = [16],         # List of resolutions with self-attention.
        dropout             = 0.10,         # Dropout probability of intermediate activations.
        label_dropout       = 0,            # Dropout probability of class labels for classifier-free guidance.

        embedding_type      = 'positional', # Timestep embedding type: 'positional' for DDPM++, 'fourier' for NCSN++.
        channel_mult_noise  = 1,            # Timestep embedding size: 1 for DDPM++, 2 for NCSN++.
        encoder_type        = 'standard',   # Encoder architecture: 'standard' for DDPM++, 'residual' for NCSN++.
        decoder_type        = 'standard',   # Decoder architecture: 'standard' for both DDPM++ and NCSN++.
        resample_filter     = [1,1],        # Resampling filter: [1,1] for DDPM++, [1,3,3,1] for NCSN++.
    ):
        assert embedding_type in ['fourier', 'positional']
        assert encoder_type in ['standard', 'skip', 'residual']
        assert decoder_type in ['standard', 'skip']

        super().__init__()
        self.label_dropout = label_dropout
        emb_channels = model_channels * channel_mult_emb
        noise_channels = model_channels * channel_mult_noise
        init = dict(init_mode='xavier_uniform')
        init_zero = dict(init_mode='xavier_uniform', init_weight=1e-5)
        init_attn = dict(init_mode='xavier_uniform', init_weight=np.sqrt(0.2))
        block_kwargs = dict(
            emb_channels=emb_channels, num_heads=1, dropout=dropout, skip_scale=np.sqrt(0.5), eps=1e-6,
            resample_filter=resample_filter, resample_proj=True, adaptive_scale=False,
            init=init, init_zero=init_zero, init_attn=init_attn,
        )

        # Mapping.
        self.map_noise = PositionalEmbedding(num_channels=noise_channels, endpoint=True) if embedding_type == 'positional' else FourierEmbedding(num_channels=noise_channels)
        self.map_label = Linear(in_features=label_dim, out_features=noise_channels, **init) if label_dim else None
        self.map_augment = Linear(in_features=augment_dim, out_features=noise_channels, bias=False, **init) if augment_dim else None
        self.map_layer0 = Linear(in_features=noise_channels, out_features=emb_channels, **init)
        self.map_layer1 = Linear(in_features=emb_channels, out_features=emb_channels, **init)

        # Encoder.
        self.enc = torch.nn.ModuleDict()
        cout = in_channels
        caux = in_channels
        for level, mult in enumerate(channel_mult):
            res = img_resolution >> level
            if level == 0:
                cin = cout
                cout = model_channels
                self.enc[f'{res}x{res}_conv'] = Conv2d(in_channels=cin, out_channels=cout, kernel=3, **init)
            else:
                self.enc[f'{res}x{res}_down'] = UNetBlock(in_channels=cout, out_channels=cout, down=True, **block_kwargs)
                if encoder_type == 'skip':
                    self.enc[f'{res}x{res}_aux_down'] = Conv2d(in_channels=caux, out_channels=caux, kernel=0, down=True, resample_filter=resample_filter)
                    self.enc[f'{res}x{res}_aux_skip'] = Conv2d(in_channels=caux, out_channels=cout, kernel=1, **init)
                if encoder_type == 'residual':
                    self.enc[f'{res}x{res}_aux_residual'] = Conv2d(in_channels=caux, out_channels=cout, kernel=3, down=True, resample_filter=resample_filter, fused_resample=True, **init)
                    caux = cout
            for idx in range(num_blocks):
                cin = cout
                cout = model_channels * mult
                attn = (res in attn_resolutions)
                self.enc[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
        skips = [block.out_channels for name, block in self.enc.items() if 'aux' not in name]

        # Decoder.
        self.dec = torch.nn.ModuleDict()
        for level, mult in reversed(list(enumerate(channel_mult))):
            res = img_resolution >> level
            if level == len(channel_mult) - 1:
                self.dec[f'{res}x{res}_in0'] = UNetBlock(in_channels=cout, out_channels=cout, attention=True, **block_kwargs)
                self.dec[f'{res}x{res}_in1'] = UNetBlock(in_channels=cout, out_channels=cout, **block_kwargs)
            else:
                self.dec[f'{res}x{res}_up'] = UNetBlock(in_channels=cout, out_channels=cout, up=True, **block_kwargs)
            for idx in range(num_blocks + 1):
                cin = cout + skips.pop()
                cout = model_channels * mult
                attn = (idx == num_blocks and res in attn_resolutions)
                self.dec[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
            if decoder_type == 'skip' or level == 0:
                if decoder_type == 'skip' and level < len(channel_mult) - 1:
                    self.dec[f'{res}x{res}_aux_up'] = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=0, up=True, resample_filter=resample_filter)
                self.dec[f'{res}x{res}_aux_norm'] = GroupNorm(num_channels=cout, eps=1e-6)
                self.dec[f'{res}x{res}_aux_conv'] = Conv2d(in_channels=cout, out_channels=out_channels, kernel=3, **init_zero)

    def forward(self, x, noise_labels, class_labels, augment_labels=None):
        # Mapping.
        emb = self.map_noise(noise_labels)
        emb = emb.reshape(emb.shape[0], 2, -1).flip(1).reshape(*emb.shape) # swap sin/cos
        if self.map_label is not None:
            tmp = class_labels
            if self.training and self.label_dropout:
                tmp = tmp * (torch.rand([x.shape[0], 1], device=x.device) >= self.label_dropout).to(tmp.dtype)
            emb = emb + self.map_label(tmp * np.sqrt(self.map_label.in_features))
        if self.map_augment is not None and augment_labels is not None:
            emb = emb + self.map_augment(augment_labels)
        emb = silu(self.map_layer0(emb))
        emb = silu(self.map_layer1(emb))

        # Encoder.
        skips = []
        aux = x
        for name, block in self.enc.items():
            if 'aux_down' in name:
                aux = block(aux)
            elif 'aux_skip' in name:
                x = skips[-1] = x + block(aux)
            elif 'aux_residual' in name:
                x = skips[-1] = aux = (x + block(aux)) / np.sqrt(2)
            else:
                x = block(x, emb) if isinstance(block, UNetBlock) else block(x)
                skips.append(x)

        # Decoder.
        aux = None
        tmp = None
        for name, block in self.dec.items():
            if 'aux_up' in name:
                aux = block(aux)
            elif 'aux_norm' in name:
                tmp = block(x)
            elif 'aux_conv' in name:
                tmp = block(silu(tmp))
                aux = tmp if aux is None else tmp + aux
            else:
                if x.shape[1] != block.in_channels:
                    x = torch.cat([x, skips.pop()], dim=1)
                x = block(x, emb)
        return aux

In [ ]:
#----------------------------------------------------------------------------
class EDMPrecond(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution.
        img_channels,                       # Number of color channels.
        label_dim       = 0,                # Number of class labels, 0 = unconditional.
        use_fp16        = False,            # Execute the underlying model at FP16 precision?
        sigma_min       = 0,                # Minimum supported noise level.
        sigma_max       = float('inf'),     # Maximum supported noise level.
        sigma_data      = 0.5,              # Expected standard deviation of the training data.
        model_type      = 'DhariwalUNet',   # Class name of the underlying model.
        **model_kwargs,                     # Keyword arguments for the underlying model.
    ):
        super().__init__()
        self.img_resolution = img_resolution
        self.img_channels = img_channels
        self.label_dim = label_dim
        self.use_fp16 = use_fp16
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.sigma_data = sigma_data
        self.model = globals()[model_type](img_resolution=img_resolution, in_channels=img_channels, out_channels=img_channels, label_dim=label_dim, **model_kwargs)

    def forward(self, x, sigma, class_labels=None, force_fp32=False, **model_kwargs):
        x = x.to(torch.float32)
        sigma = sigma.to(torch.float32).reshape(-1, 1, 1, 1)
        class_labels = None if self.label_dim == 0 else torch.zeros([1, self.label_dim], device=x.device) if class_labels is None else class_labels.to(torch.float32).reshape(-1, self.label_dim)
        dtype = torch.float16 if (self.use_fp16 and not force_fp32 and x.device.type == 'cuda') else torch.float32

        c_skip = self.sigma_data ** 2 / (sigma ** 2 + self.sigma_data ** 2) #1
        c_out = sigma * self.sigma_data / (sigma ** 2 + self.sigma_data ** 2).sqrt() #0
        c_in = 1 / (self.sigma_data ** 2 + sigma ** 2).sqrt()
        c_noise = sigma.log() / 4

        F_x = self.model((c_in * x).to(dtype), c_noise.flatten(), class_labels=class_labels, **model_kwargs)
        assert F_x.dtype == dtype
        D_x = c_skip * x + c_out * F_x.to(torch.float32)
        return D_x

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

### Network config and verification

In [ ]:
# main.py:50-57, with img_resolution and attn_resolutions carrying the data change.
CHANNEL_MULT = [2, 2, 2]
ATTN_RES = [GRID >> (len(CHANNEL_MULT) - 1)]    # deepest level: 128>>2 = 32  (main.py: 64>>2 = 16)
assert ATTN_RES == [32], ATTN_RES

NET_KWARGS = dict(
    img_resolution   = GRID,        # main.py:55 has 64; our fields are 128
    img_channels     = 1,
    label_dim        = 0,
    use_fp16         = False,
    model_type       = 'SongUNet',
    embedding_type   = 'positional',
    encoder_type     = 'standard',
    decoder_type     = 'standard',
    channel_mult_noise = 1,
    resample_filter  = [1, 1],
    channel_mult     = CHANNEL_MULT,
    dropout          = 0.0,
    attn_resolutions = ATTN_RES,    # main.py leaves the SongUNet default [16]; see markdown above
)
# num_blocks=4 is a SongUNet default; main.py does not override it.

MODEL_CHANNELS = 16   # 880,097 params -- cheapest capacity that fully memorizes at n=2

def build_net(model_channels, device=None):
    net = EDMPrecond(model_channels=model_channels, **NET_KWARGS)
    return net if device is None else net.to(device)

def count_params(net):
    return sum(p.numel() for p in net.parameters())

PAPER_FIG18_PARAMS = {4: 57017, 8: 222705, 16: 880097,
                      32: 3498945, 64: 13952897, 128: 55725825}

n = count_params(build_net(MODEL_CHANNELS))
assert n == PAPER_FIG18_PARAMS[MODEL_CHANNELS], (
    f'model_channels={MODEL_CHANNELS}: got {n:,}, Figure 18 says {PAPER_FIG18_PARAMS[MODEL_CHANNELS]:,}')
print(f'model_channels={MODEL_CHANNELS}: {n:,} params (matches Figure 18)')

## The loss -- verbatim

`training/loss.py:65-81`. `main.py:63-65` constructs `EDMLoss` with no arguments, so `P_mean=-1.2`,
`P_std=1.2`, `sigma_data=0.5` are all class defaults.

In [ ]:
class EDMLoss:
    """training/loss.py, verbatim. Returns the per-element loss; the caller reduces it."""
    def __init__(self, P_mean=-1.2, P_std=1.2, sigma_data=0.5):
        self.P_mean = P_mean
        self.P_std = P_std
        self.sigma_data = sigma_data

    def __call__(self, net, images, labels=None, augment_pipe=None):
        rnd_normal = torch.randn([images.shape[0], 1, 1, 1], device=images.device)
        sigma = (rnd_normal * self.P_std + self.P_mean).exp()
        weight = (sigma ** 2 + self.sigma_data ** 2) / (sigma * self.sigma_data) ** 2
        y, augment_labels = augment_pipe(images) if augment_pipe is not None else (images, None)
        n = torch.randn_like(y) * sigma
        D_yn = net(y + n, sigma, labels, augment_labels=augment_labels)
        loss = weight * ((D_yn - y) ** 2)
        return loss

## The sampler -- verbatim

`generate.py:25-60`, EDM Algorithm 2; `main.py:134` leaves `S_churn=0`, so no noise is injected and
Algorithm 2 degenerates to deterministic 2nd-order Heun -- an ODE.

In [ ]:
# float64 everywhere, exactly as EDM -- except on MPS, which cannot allocate float64 at all.
SAMPLER_DTYPE = torch.float32 if DEVICE.type == 'mps' else torch.float64
if SAMPLER_DTYPE is torch.float32:
    print('WARNING: MPS cannot do float64; sampling in float32. EDM (and a CUDA run) uses '
          'float64 -- do not report numbers from an MPS run.')


def edm_sampler(
    net, latents, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
    """generate.py, verbatim (EDM Algorithm 2)."""
    # Adjust noise levels based on what's supported by the network.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    # Time step discretization.
    step_indices = torch.arange(num_steps, dtype=SAMPLER_DTYPE, device=latents.device)  # EDM: torch.float64
    t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

    # Main sampling loop.
    x_next = latents.to(SAMPLER_DTYPE) * t_steps[0]  # EDM: torch.float64
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
        x_cur = x_next

        # Increase noise temporarily.
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

        # Euler step.
        denoised = net(x_hat, t_hat, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        # Apply 2nd order correction.
        if i < num_steps - 1:
            denoised = net(x_next, t_next, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

    return x_next

## Configuration

In [ ]:
# Few-minute end-to-end check of the whole pipeline:  FIELD_SMOKE=1 python3 -m nbconvert ...
SMOKE = os.environ.get('FIELD_SMOKE', '0').strip().lower() not in ('0', '', 'false', 'no')

CFG = dict(
    # -- from main.py, verbatim --------------------------------------------------------------
    epochs             = 50_000,   # main.py:15
    batch_mode         = 'main_py',# DataLoader(batch_size=1, shuffle=True), 2 updates/epoch
    lr                 = 10e-4,    # main.py:61
    betas              = (0.9, 0.999),
    eps                = 1e-8,
    lr_rampup_kimg     = 10_000,   # main.py:19  -> lr never exceeds 1e-5 over this run
    ema_halflife_kimg  = 500,      # main.py:20
    ema_rampup_ratio   = 0.05,     # main.py:21
    P_mean             = -1.2,     # EDMLoss defaults, main.py:63-65
    P_std              = 1.2,
    sigma_data         = 0.5,      # networks.py:640 default; NOT estimated from the data

    # -- sampler: main.py:134 calls edm_sampler(ema, latents, num_steps=40), rest defaulted ----
    num_steps          = 40,
    sigma_min          = 0.002,
    sigma_max          = 80.0,     # per rescaled geometry (GEOM_SPECS in source notebook)
    rho                = 7,
    S_churn            = 0.0,      # deterministic Heun (ODE)
    S_min              = 0.0,
    S_max              = float('inf'),
    S_noise            = 1.0,

    # -- evaluation protocol (matching source notebook) ----------------------------------------
    n_eval_samples     = 100,
    eval_every         = 1_000,    # every 1000 epochs
    rel_threshold      = 0.3,      # collapse threshold on relative distance

    # -- ring metric -------------------------------------------------------------------------
    n_rand_ref         = 32,       # random reference draws for memorization ratio

    # -- run mechanics ------------------------------------------------------------------------
    seed               = 0,
    latent_seed        = 42,
    eval_batch         = 25,       # 128^2 in float64; lower this first on an OOM
    save_resume_state  = True,
    n_sample_grid      = 16,
)

if SMOKE:
    CFG.update(epochs=300, eval_every=100, n_eval_samples=16, num_steps=10,
               eval_batch=16, save_resume_state=False, n_rand_ref=8)

# Output directories
RESULT_DIR = os.path.join(results_dir, 'songunet_cov_tikhonov')
os.makedirs(RESULT_DIR, exist_ok=True)
STATE_DIR = RESULT_DIR

def arm_result_path(variant, c_val):
    return os.path.join(RESULT_DIR, f'arm_{variant}_c{c_val:g}_result.pt')

def arm_state_path(variant, c_val):
    return os.path.join(STATE_DIR, f'arm_{variant}_c{c_val:g}.pt')

print(f"{'SMOKE RUN' if SMOKE else 'FULL RUN'}")
print(f"  model_channels : {MODEL_CHANNELS}  ({PAPER_FIG18_PARAMS[MODEL_CHANNELS]:,} params)")
print(f"  epochs         : {CFG['epochs']:,}  (batch_mode={CFG['batch_mode']}, "
      f"{CFG['epochs'] * (N_TRAIN if CFG['batch_mode'] == 'main_py' else 1):,} optimizer updates)")
print(f"  sigma_max      : {CFG['sigma_max']}")
print(f"  sigma_data     : {CFG['sigma_data']}")
print(f"  results dir    : {RESULT_DIR}")

## The training data

The first two draws of the standard 200-field multiband pool (`seed=42`, `normalize=True`),
rescaled so $D_-$ matches Baptista's rectangle pair. Identical to `baptista_config_matern_n2.ipynb`
geometry='rescaled'.

In [ ]:
from multiband_data_utils import generate_multiband_dataset_postmask, make_knrm_grid

# The standard pool -- identical call to every other field notebook in this folder.
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
weights = [1.0, 0.8, 0.8, 1.2]

_pool = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=GRID, components=components,
    weights=weights, seed=42, normalize=True,
)
normalization_std = _pool['normalization']['std']
_base = _pool['combined'][:N_TRAIN].reshape(N_TRAIN, 1, GRID, GRID).clone().float()

def _pair_dist(x):
    f = x.reshape(x.shape[0], -1)
    return torch.cdist(f, f)[0, 1].item()

RECT_D_MINUS = math.sqrt(340.0)
D_UNIT = _pair_dist(_base)
SCALE = RECT_D_MINUS / D_UNIT

data = _base * SCALE
D_MINUS = _pair_dist(data)

# Hard guard: this IS the rescaled geometry from the source notebook
assert abs(D_MINUS - RECT_D_MINUS) < 1e-3, f'D_minus={D_MINUS:.4f}, expected {RECT_D_MINUS:.4f}'

print(f'N_TRAIN={N_TRAIN}, GRID={GRID}')
print(f'D_UNIT={D_UNIT:.4f}, scale={SCALE:.5f}')
print(f'D_minus (rescaled) = {D_MINUS:.4f}  (target: {RECT_D_MINUS:.4f})')
print(f'sigma_max / D_minus = {CFG["sigma_max"] / D_MINUS:.4f}')
print(f'rescaled data std = {data.std():.5f}')
print(f'normalization_std = {normalization_std:.6f}')

## Spectrum estimation and Tikhonov penalty

All Tikhonov logic is notebook-local. No `src/` changes.

**Reduction mismatch fix.** Baptista's training loop (`main.py`) uses `loss = loss_fn(net, images, ...).sum() / batch_size`
-- that is a SUM over $d=128^2=16384$ pixels per sample. The Tikhonov penalty must use the same
reduction. The penalty function below returns a per-sample scalar that is already a SUM over the
$d$ pixel/mode dimensions, matching `EDMLoss.__call__` which returns `weight * (D_yn - y)**2`
with shape `(B,1,N,N)`, and the training loop reduces via `.sum() / B`.

In [ ]:
# ---------------------------------------------------------------------------
# Spectrum estimation: empirical (multiple n) and population
# ---------------------------------------------------------------------------

knrm = make_knrm_grid(GRID).double()
ring = knrm.round().long()
nring = int(ring.max()) + 1
cnt = torch.zeros(nring, dtype=torch.float64).index_add_(
    0, ring.flatten(), torch.ones(GRID * GRID, dtype=torch.float64))

def radial_avg_spectrum(fields_2d):
    """Uncentered radially-averaged periodogram from (n, H, W) fields."""
    P = (torch.fft.fft2(fields_2d.double(), norm='forward').abs() ** 2).mean(0)
    tot = torch.zeros(nring, dtype=torch.float64).index_add_(0, ring.flatten(), P.flatten())
    radial = tot / cnt.clamp_min(1)
    return radial[ring]  # broadcast back to (GRID, GRID)

# -- Empirical spectra at different pool sizes --
# Training data is _pool['combined'][:2], rescaled. For n>2 we draw more from the pool,
# apply the same rescaling, and estimate the spectrum. These extra fields are NOT used
# for training — only for spectrum estimation.
_pool_all = _pool['combined'].double()  # (200, GRID, GRID), normalized but unrescaled
ESTIMATION_NS = [2, 8, 16, 32]
lam_emp = {}
for n_est in ESTIMATION_NS:
    fields = _pool_all[:n_est] * SCALE  # rescale same as training data
    lam_emp[n_est] = radial_avg_spectrum(fields)
    err_tag = f'n={n_est}'
    print(f'  {err_tag}: estimated from {n_est} fields')

# -- Population: exact analytic spectrum --
lam_pop = torch.zeros(GRID, GRID, dtype=torch.float64)
for comp, w in zip(components, weights):
    S = comp['sigma_sq'] * (knrm**2 + comp['length_scale']**2) ** (-comp['s'])
    S[0, 0] = 0.0
    k_lo, k_hi = comp['band']
    S = S * ((knrm >= k_lo) & (knrm < k_hi)).double()
    lam_pop += (w**2) * S
lam_pop = lam_pop / (normalization_std ** 2) * (SCALE ** 2)

# -- Null mode handling --
NULL_REL_THRESHOLD = 1e-4

def build_inv_lam(lam_2d):
    active = lam_2d > NULL_REL_THRESHOLD * lam_2d[lam_2d > 0].mean()
    inv = torch.where(active, 1.0 / lam_2d.clamp_min(1e-30), torch.zeros_like(lam_2d))
    return inv, active

inv_lam_emp = {}
for n_est in ESTIMATION_NS:
    inv_lam_emp[n_est], _ = build_inv_lam(lam_emp[n_est])

inv_lam_pop, active_pop = build_inv_lam(lam_pop)

# Budget-match: scale each inv_lam so sum = M^2, making c comparable to isotropic.
# Forward-norm Parseval: sum|FFT_fwd|^2 = (1/M)*sum|x|^2. The isotropic penalty sums
# pixels (budget M), so the covariance penalty needs inv_lam summing to M^2 to compensate.
M = GRID * GRID
for n_est in ESTIMATION_NS:
    s = inv_lam_emp[n_est].sum()
    if s > 0:
        inv_lam_emp[n_est] = inv_lam_emp[n_est] * (M * M / s)
s = inv_lam_pop.sum()
if s > 0:
    inv_lam_pop = inv_lam_pop * (M * M / s)
print(f'Budget-matched: each inv_lam scaled so sum = {M*M} (= M^2, compensates forward-norm Parseval)')

# Diagnostics: estimation error vs population
print(f'\nPopulation spectrum: {active_pop.sum().item()}/{GRID*GRID} active modes')
for n_est in ESTIMATION_NS:
    _, active_n = build_inv_lam(lam_emp[n_est])
    both = active_n & active_pop
    rel = ((lam_emp[n_est][both] - lam_pop[both]).abs() / lam_pop[both]).mean()
    budget = ((1.0 / lam_emp[n_est][both]).sum() / (1.0 / lam_pop[both]).sum()).item()
    print(f'  n={n_est:>2}: mean |err| = {rel:.1%}, budget ratio = {budget:.2f}x')


In [ ]:
# ---------------------------------------------------------------------------
# Tikhonov penalty (notebook-local)
# ---------------------------------------------------------------------------

def tikhonov_penalty_cov(D_theta, x_noisy, sigma, c, inv_lam_2d):
    """Covariance-weighted Tikhonov penalty on the implied score.

    D_theta: denoiser output (B, 1, N, N)
    x_noisy: noisy input (B, 1, N, N)
    sigma: noise level (B,) or scalar
    c: regularization constant
    inv_lam_2d: 1/lambda(k), shape (N, N), with null modes zeroed

    Score = (D_theta - x_noisy) / sigma^2  (EDM preconditioning)
    Penalty per sample = c/sigma^2 * sum_k |score_hat_k|^2 / lambda(k)
           = c/sigma^4 * sum_k |FFT(D_theta - x_noisy)_k|^2 / lambda(k)

    The 'forward' FFT norm gives coefficients with the standard 1/N^2 scaling.
    We use SUM over modes (not mean) to match EDMLoss's sum-over-pixels reduction.

    Returns: (B,) per-sample penalty.
    """
    s2 = sigma.reshape(-1, 1, 1, 1) ** 2
    diff = D_theta - x_noisy  # (B, 1, N, N)
    diff_hat = torch.fft.fft2(diff.squeeze(1), norm='forward')  # (B, N, N)
    # Per-mode weighted: |diff_hat_k|^2 * (1/lambda(k))
    per_mode = diff_hat.abs() ** 2 * inv_lam_2d.unsqueeze(0).to(diff_hat.device)
    # Sum over spatial modes (matching sum-over-pixels in the data term)
    penalty_per_sample = per_mode.sum(dim=(-2, -1))  # (B,)
    # Divide by sigma^4 (score = diff/sigma^2, so |score|^2 ~ diff^2/sigma^4),
    # then multiply by c. But we want c/sigma^2 * sum|score_hat|^2 / lambda
    # = c/sigma^2 * sum|diff_hat/sigma^2|^2 / lambda = c/sigma^4 * sum|diff_hat|^2/lambda
    # Actually, let's rewrite:
    # The EDM loss weight already contains weight(sigma), and in Baptista's formulation
    # the Tikhonov penalty carries the same weight lambda(sigma) as the data term.
    # So the penalty here should be: c * ||D - x||^2_{inv_lam} / sigma^2
    # (analogous to src/edm.py:tikhonov_penalty which uses c * ((D-x)^2 / sigma^2).mean())
    # With sum reduction: c * sum_k |FFT(D-x)_k|^2 / (sigma^2 * lambda(k))
    return (c / s2.squeeze()) * penalty_per_sample


def tikhonov_penalty_iso(D_theta, x_noisy, sigma, c):
    """Isotropic Tikhonov penalty: c/sigma^2 * sum_pixels (D_theta - x_noisy)^2.

    Returns: (B,) per-sample penalty.
    """
    s2 = sigma.reshape(-1, 1, 1, 1) ** 2
    diff_sq = (D_theta - x_noisy) ** 2  # (B, 1, N, N)
    # Sum over all pixels (matching EDMLoss reduction)
    return c * diff_sq.sum(dim=(1, 2, 3)) / s2.squeeze()


print('Tikhonov penalty functions defined (notebook-local, sum reduction)')

## c grid calibration

The GMM study used $c \in \{10^{-4}, 10^{-3}, 10^{-2}, 0.1, 1.0, 5.0\}$ on unrescaled data.
Rescaling multiplies $\lambda$ by scale$^2 \approx 0.00984$, and the sampler is different
(Heun ODE vs SDE), so those values do not transfer directly.

We calibrate by computing the penalty/data-term ratio at a representative sigma for each
candidate $c$, using the closed-form isotropic Tikhonov score. Pick 4 values spanning from
"barely regularized" to "heavily regularized".

In [ ]:
# Calibration: what fraction of the total loss comes from the Tikhonov penalty
# at a representative sigma, for each candidate c?
#
# At the stationary point of isotropic Tikhonov, the denoiser output is:
#   D_theta = (sigma^2 * x_0 + c * x_noisy) / (sigma^2 + c)
# and the implied score is s = (D-x) / sigma^2 = -x_0 / (sigma^2 + c) + noise/sigma^2 * c/(sigma^2+c)
# The data-term residual D-x_0 = c*(x_noisy - x_0)/(sigma^2 + c) = c*noise*sigma/(sigma^2+c)
# The penalty-term residual D-x_noisy = sigma^2*(x_0 - x_noisy)/(sigma^2+c) = -sigma^2*noise*sigma/(sigma^2+c)
#
# So at the stationary point:
#   data_loss ~ weight * c^2 * sigma^2 / (sigma^2+c)^2 * E[noise^2]
#   penalty   ~ weight * c/sigma^2 * sigma^4 * sigma^2 / (sigma^2+c)^2 * E[noise^2]
#             = weight * c * sigma^4 / (sigma^2+c)^2 * E[noise^2]
#   ratio = penalty / data_loss = sigma^2 / c
#
# This means: at sigma_representative, penalty/data ~ sigma_rep^2 / c.
# P_mean = -1.2, so the median training sigma = exp(-1.2) ~ 0.30.

sigma_rep = math.exp(CFG['P_mean'])  # median training sigma
print(f'Representative sigma (median of training distribution): {sigma_rep:.4f}')
print(f'sigma_rep^2 = {sigma_rep**2:.6f}')
print()

# Candidate c values and their penalty/data ratios
c_candidates = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0, 3.0]
print(f'{"c":>10s}  {"pen/data":>10s}  {"c/(sigma^2+c)":>14s}  note')
print('-' * 60)
for c in c_candidates:
    ratio = sigma_rep**2 / c
    shrinkage = c / (sigma_rep**2 + c)
    note = ''
    if 0.005 < ratio < 0.02:
        note = '<-- barely regularized'
    elif 0.08 < ratio < 0.15:
        note = '<-- mild'
    elif 0.4 < ratio < 0.6:
        note = '<-- moderate'
    elif 1.5 < ratio < 2.5:
        note = '<-- heavy'
    print(f'{c:>10.1e}  {ratio:>10.3f}  {shrinkage:>14.4f}  {note}')

# Select 4 values that span barely -> heavy regularization
# Aiming for pen/data ~ 0.01, 0.1, 0.5, 2.0
# At sigma_rep^2 ~ 0.09: c ~ 9, 0.9, 0.18, 0.045
# Round to clean values:
C_VALUES = [3e-3, 1e-2, 3e-2, 0.1]

if SMOKE:
    C_VALUES = [1e-2]

print(f'\nSelected c values: {C_VALUES}')
for c in C_VALUES:
    ratio = sigma_rep**2 / c
    print(f'  c={c:g}: penalty/data ratio at median sigma = {ratio:.2f}')

## The metric

In [ ]:
from memorization_metrics import RingMetricContext, radial_power_spectrum

bands = {c['name']: c['band'] for c in components}
ctx = RingMetricContext(GRID, bands, device=DEVICE)

@torch.no_grad()
def field_stats(x_gen, x_train, rel_threshold=0.3):
    '''main.py:150-159 L2-to-data-manifold, plus a scale-free relative version.'''
    g = x_gen.detach().float().cpu().reshape(x_gen.shape[0], -1)
    t = x_train.detach().float().cpu().reshape(x_train.shape[0], -1)
    d2 = torch.cdist(g, t) ** 2
    l2_min = d2.min(dim=1).values
    nn_idx = d2.argmin(dim=1)
    nn_rel = l2_min.clamp_min(0).sqrt() / t.norm(dim=1).mean()
    return {
        'l2_max':        l2_min.max().item(),
        'l2_mean':       l2_min.mean().item(),
        'l2_min':        l2_min.min().item(),
        'nn_rel_median': nn_rel.median().item(),
        'nn_rel_min':    nn_rel.min().item(),
        'fraction':      (nn_rel < rel_threshold).float().mean().item(),
        'nn_index':      nn_idx,
    }


@torch.no_grad()
def ring_metric_eval(x_gen_2d, x_train_2d):
    '''Per-band ring metric evaluation with exclude_nn=True.'''
    m = ctx.evaluate(
        x_gen_2d.to(DEVICE), x_train_2d.to(DEVICE),
        n_rand_ref=CFG['n_rand_ref'], exclude_nn=True,
        aggregate='mean_of_ratios',
    )
    return {
        'mean_ratio': m['mean_ratio'].cpu(),
        'coarse_score': m['coarse_score'].mean().item(),
        'mid1_score': m['mid1_score'].mean().item(),
        'mid2_score': m['mid2_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
    }


# Sanity check: training data against itself
_self = field_stats(data, data)
assert _self['l2_max'] < 1e-8 and _self['fraction'] == 1.0, _self
print(f'metric check: training data -> fraction {_self["fraction"]:.2f}')

## The gate -- can this sampler reach the memorizing solution?

Closed-form empirical-Bayes denoiser (fully memorizing by construction) through the same
`edm_sampler`. Must pass for the training results to be interpretable.

In [ ]:
class GMMDenoiser(torch.nn.Module):
    '''Exact empirical-Bayes denoiser for the N-point empirical measure = full memorization.'''
    sigma_min = 0.0
    sigma_max = float('inf')

    def __init__(self, y):
        super().__init__()
        self.register_buffer('y', y.reshape(y.shape[0], -1))

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

    def forward(self, x, sigma, class_labels=None):
        shp = x.shape
        xf = x.reshape(shp[0], -1).to(self.y.dtype)
        d2 = torch.cdist(xf, self.y) ** 2
        s = torch.as_tensor(sigma, dtype=self.y.dtype, device=xf.device).reshape(-1, 1)
        w = torch.softmax(-d2 / (2 * s ** 2), dim=1)
        return (w @ self.y).reshape(shp)


@torch.no_grad()
def run_gate(data_tensor, cfg):
    '''Push the closed-form memorizing denoiser through this sampler.'''
    g = torch.Generator().manual_seed(cfg['latent_seed'])
    lat = torch.randn(cfg['n_sample_grid'], 1, GRID, GRID, generator=g).to(DEVICE)
    gmm = GMMDenoiser(data_tensor.to(DEVICE).to(SAMPLER_DTYPE))
    xg = edm_sampler(gmm, lat, num_steps=cfg['num_steps'],
                     sigma_min=cfg['sigma_min'], sigma_max=cfg['sigma_max'], rho=cfg['rho'],
                     S_churn=cfg['S_churn'], S_min=cfg['S_min'], S_max=cfg['S_max'],
                     S_noise=cfg['S_noise'])
    st = field_stats(xg.float().cpu(), data_tensor, rel_threshold=cfg['rel_threshold'])
    st.pop('nn_index')
    st['passed'] = st['nn_rel_median'] < 0.05
    return st


GATE = run_gate(data, CFG)
print(f"closed-form memorizing denoiser through the same sampler:")
print(f"  L2 to manifold  max {GATE['l2_max']:.4e}   mean {GATE['l2_mean']:.4e}")
print(f"  nn_rel median   {GATE['nn_rel_median']:.5f}")
print(f"  fraction < {CFG['rel_threshold']}   {GATE['fraction']:.2f}")
if GATE['passed']:
    print('  GATE PASSED -- the memorizing solution is reachable by this sampler.')
else:
    print('  GATE FAILED -- even the exact memorizing score does not produce collapsed')
    print('                 samples. Non-collapse below is a sampler statement, not a network one.')

## Arm definitions

25 arms total: 1 gate (c=0), 4 isotropic, 4x4 covariance-empirical (n=2,8,16,32), 4 covariance-population.

Training is always on the same 2 fields. The covariance arms differ only in how many
fields from the 200-field pool are used to estimate the spectrum for the regularizer.


In [ ]:
# Build the arm list
ARMS = []

# 1. Gate arm: c=0, unregularized
ARMS.append(dict(variant='gate', c=0.0, inv_lam=None, label='gate (c=0)'))

# 2. Isotropic Tikhonov
for c in C_VALUES:
    ARMS.append(dict(variant='isotropic', c=c, inv_lam=None, label=f'iso c={c:g}'))

# 3. Covariance-empirical at different estimation pool sizes
for n_est in ESTIMATION_NS:
    for c in C_VALUES:
        ARMS.append(dict(
            variant=f'cov_emp_n{n_est}', c=c,
            inv_lam=inv_lam_emp[n_est].float().to(DEVICE),
            label=f'cov_emp_n{n_est} c={c:g}',
        ))

# 4. Covariance-population
for c in C_VALUES:
    ARMS.append(dict(
        variant='cov_population', c=c,
        inv_lam=inv_lam_pop.float().to(DEVICE),
        label=f'cov_pop c={c:g}',
    ))

print(f'{len(ARMS)} arms total:')
for arm in ARMS:
    print(f'  {arm["label"]}')


## Training and evaluation

`main.py:73-107`, transcribed from `baptista_config_matern_n2.ipynb`. The only additions are
the Tikhonov penalty in the loss and per-band ring metric evaluation at each checkpoint.

Idempotent: completed arms are skipped, partial ones resume from state files.

In [ ]:
@torch.no_grad()
def generate_samples(ema_net, n_samples, cfg, device, seed):
    '''Draw n_samples with the EDM sampler, chunked to fit in memory.'''
    ema_net.eval()
    out = []
    remaining = n_samples
    chunk_i = 0
    while remaining > 0:
        b = min(cfg['eval_batch'], remaining)
        g = torch.Generator().manual_seed(seed + 1000 * chunk_i)
        latents = torch.randn(b, 1, GRID, GRID, generator=g).to(device)
        x = edm_sampler(ema_net, latents,
                        num_steps=cfg['num_steps'], sigma_min=cfg['sigma_min'],
                        sigma_max=cfg['sigma_max'], rho=cfg['rho'], S_churn=cfg['S_churn'],
                        S_min=cfg['S_min'], S_max=cfg['S_max'], S_noise=cfg['S_noise'])
        out.append(x.float().cpu())
        remaining -= b
        chunk_i += 1
    return torch.cat(out, dim=0)


def run_arm(arm, cfg, device, resume=True, log=print):
    '''One training arm. Returns the evaluation log.'''
    variant, c_val = arm['variant'], arm['c']
    inv_lam_device = arm['inv_lam']  # None for gate/isotropic, (GRID,GRID) tensor for cov
    state_path = arm_state_path(variant, c_val)

    torch.manual_seed(cfg['seed'])
    net = build_net(MODEL_CHANNELS, device)
    net.train().requires_grad_(True)
    ema = copy.deepcopy(net).eval().requires_grad_(False)
    optimizer = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                                 betas=list(cfg['betas']), eps=cfg['eps'])

    cur_nimg, opt_step, start_epoch = 1, 0, 0
    eval_log, loss_hist = [], []

    if resume and os.path.exists(state_path):
        st = torch.load(state_path, map_location='cpu', weights_only=False)
        net.load_state_dict(st['net']); ema.load_state_dict(st['ema'])
        optimizer.load_state_dict(st['opt'])
        cur_nimg, opt_step, start_epoch = st['cur_nimg'], st['opt_step'], st['epoch']
        eval_log, loss_hist = st['eval_log'], st['loss_hist']
        torch.set_rng_state(st['rng_cpu'])
        if st.get('rng_cuda') is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(st['rng_cuda'])
        if st.get('rng_mps') is not None and torch.backends.mps.is_available():
            torch.mps.set_rng_state(st['rng_mps'])
        log(f'  resumed from epoch {start_epoch:,}')

    x_train_cpu = data
    x_train_2d = data.squeeze(1)  # (2, 128, 128) for ring metric

    # Baptista's batch_mode='main_py': DataLoader(batch_size=1, shuffle=True)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(data), batch_size=1, shuffle=True)

    t_start = time.time()
    for ep in range(start_epoch, cfg['epochs']):
        err, count = 0.0, 0
        net.train()
        for (x,) in loader:
            optimizer.zero_grad(set_to_none=True)
            x = x.to(device)

            # Inline the EDMLoss computation so the Tikhonov penalty shares
            # the same sigma, noise, and D_theta as the data term. This is
            # essential: the stationary-point argument requires both terms
            # to see the same noise realization.
            # (EDMLoss.__call__ encapsulates these, so we cannot reuse it
            # and still access sigma/x_noisy/D_theta for the penalty.)
            rnd_normal = torch.randn([x.shape[0], 1, 1, 1], device=x.device)
            sigma = (rnd_normal * cfg['P_std'] + cfg['P_mean']).exp()
            weight = (sigma ** 2 + cfg['sigma_data'] ** 2) / (sigma * cfg['sigma_data']) ** 2
            n = torch.randn_like(x) * sigma
            x_noisy = x + n
            D_theta = net(x_noisy, sigma)

            # Data term: weight * ||D_theta - x_0||^2, sum over pixels, mean over batch
            # This is loss_fn(net, x).sum() / B, inlined.  (main.py:84)
            data_loss = weight * ((D_theta - x) ** 2)
            loss = data_loss.sum() / x.size(0)

            # Tikhonov penalty on the SAME forward pass
            if c_val > 0:
                if inv_lam_device is not None:
                    pen = tikhonov_penalty_cov(D_theta, x_noisy, sigma.squeeze(), c_val,
                                               inv_lam_device)
                else:  # isotropic
                    pen = tikhonov_penalty_iso(D_theta, x_noisy, sigma.squeeze(), c_val)

                # The penalty carries the same EDM weight as the data term,
                # so the stationary condition gives s* = s_true / (1 + c_eff/sigma^2).
                loss = loss + (weight.squeeze() * pen).sum() / x.size(0)

            loss.backward()

            # lr ramp-up: main.py:88-89
            for g in optimizer.param_groups:
                g['lr'] = cfg['lr'] * min(cur_nimg / max(cfg['lr_rampup_kimg'] * 1000, 1e-8), 1)
            # nan guard: main.py:91-93
            for param in net.parameters():
                if param.grad is not None:
                    torch.nan_to_num(param.grad, nan=0, posinf=1e5, neginf=-1e5, out=param.grad)
            optimizer.step()

            # EMA update: main.py:96-102
            ema_halflife_nimg = cfg['ema_halflife_kimg'] * 1000
            if cfg['ema_rampup_ratio'] is not None:
                ema_halflife_nimg = min(ema_halflife_nimg, cur_nimg * cfg['ema_rampup_ratio'])
            ema_beta = 0.5 ** (x.size(0) / max(ema_halflife_nimg, 1e-8))
            for p_ema, p_net in zip(ema.parameters(), net.parameters()):
                p_ema.copy_(p_net.detach().lerp(p_ema, ema_beta))

            err += loss.item()
            cur_nimg += x.size(0)
            count += x.size(0)
            opt_step += 1

        loss_hist.append(err / count)

        if (ep + 1) % cfg['eval_every'] == 0:
            n_done = (ep + 1) // cfg['eval_every']
            x_gen = generate_samples(ema, cfg['n_eval_samples'], cfg, device,
                                     seed=cfg['latent_seed'] + n_done)
            # Collapse metric (field_stats)
            st = field_stats(x_gen, x_train_cpu, rel_threshold=cfg['rel_threshold'])

            # Per-band ring metric
            x_gen_2d = x_gen.squeeze(1)  # (n_eval, 128, 128)
            ring_m = ring_metric_eval(x_gen_2d, x_train_2d)

            row = dict(
                epoch=ep + 1, opt_step=opt_step, cur_nimg=cur_nimg,
                lr=optimizer.param_groups[0]['lr'],
                loss=float(np.mean(loss_hist[-cfg['eval_every']:])),
                l2_max=st['l2_max'], l2_mean=st['l2_mean'], l2_min=st['l2_min'],
                nn_rel_median=st['nn_rel_median'], nn_rel_min=st['nn_rel_min'],
                fraction=st['fraction'],
                coarse_score=ring_m['coarse_score'],
                mid1_score=ring_m['mid1_score'],
                mid2_score=ring_m['mid2_score'],
                fine_score=ring_m['fine_score'],
                mean_ratio=ring_m['mean_ratio'],
                samples=x_gen[:cfg['n_sample_grid']].clone(),
            )
            eval_log.append(row)

            el = time.time() - t_start
            frac_done = (ep + 1 - start_epoch) / max(cfg['epochs'] - start_epoch, 1)
            eta = el / max(frac_done, 1e-9) - el
            log(f"  epoch {ep+1:>7,} | step {opt_step:>7,} | lr {row['lr']:.2e} | "
                f"loss {row['loss']:.4f} | frac {st['fraction']:.2f} | "
                f"coarse {ring_m['coarse_score']:.4f} fine {ring_m['fine_score']:.4f} | "
                f"{el/60:.1f}m elapsed, ~{eta/60:.0f}m left")

            if cfg['save_resume_state']:
                torch.save(dict(
                    net=net.state_dict(), ema=ema.state_dict(),
                    opt=optimizer.state_dict(), cur_nimg=cur_nimg,
                    opt_step=opt_step, epoch=ep + 1,
                    eval_log=eval_log, loss_hist=loss_hist,
                    rng_cpu=torch.get_rng_state(),
                    rng_cuda=(torch.cuda.get_rng_state_all()
                              if torch.cuda.is_available() else None),
                    rng_mps=(torch.mps.get_rng_state()
                             if torch.backends.mps.is_available() else None),
                ), state_path)

    return dict(
        eval_log=eval_log, loss_hist=loss_hist,
        n_params=count_params(net), model_channels=MODEL_CHANNELS,
        variant=variant, c=c_val,
        minutes=(time.time() - t_start) / 60,
    )

### Run all arms

Idempotent: a completed arm is skipped and a partial one resumes from its state file.

In [ ]:
NOTE = ('SongUNet (EDMPrecond, model_channels=16, 880097 params) with covariance-weighted '
        'Tikhonov regularization on rescaled Matern fields, n_train=2. '
        'Config transcribed verbatim from baptista_config_matern_n2.ipynb (Baptista et al. '
        'arXiv:2501.15785 section 5.4). Geometry: rescaled so D_- matches their rectangles. '
        'Tikhonov penalty is notebook-local with sum reduction matching EDMLoss. '
        'Ring metric: exclude_nn=True, mean_of_ratios, bands coarse/mid1/mid2/fine.')

n_expect = CFG['epochs'] // CFG['eval_every']

# Arm sharding: split ARMS across parallel processes so each GPU takes a subset.
# A SLURM job array sets SLURM_ARRAY_TASK_ID / SLURM_ARRAY_TASK_COUNT automatically.
# ARM_SHARD / ARM_NSHARDS override them for manual runs.
# Default (0, 1) runs every arm in one process -- unchanged single-GPU behaviour.
SHARD   = int(os.environ.get('ARM_SHARD',   os.environ.get('SLURM_ARRAY_TASK_ID', 0)))
NSHARDS = int(os.environ.get('ARM_NSHARDS', os.environ.get('SLURM_ARRAY_TASK_COUNT', 1)))
assert 0 <= SHARD < NSHARDS, f'bad shard {SHARD}/{NSHARDS}'
my_arms = ARMS[SHARD::NSHARDS]   # strided, so cost balances across shards
print(f'shard {SHARD} of {NSHARDS}: {len(my_arms)} of {len(ARMS)} arms', flush=True)
for _a in my_arms:
    print(f'  {_a["label"]}', flush=True)
print(flush=True)

for arm in my_arms:
    variant, c_val = arm['variant'], arm['c']
    p = arm_result_path(variant, c_val)
    if os.path.exists(p):
        prev = torch.load(p, map_location='cpu', weights_only=False)
        if len(prev['eval_log']) >= n_expect:
            print(f'=== {arm["label"]} already complete, skipping ===', flush=True)
            continue
    print(f'=== {arm["label"]} ===', flush=True)
    res = run_arm(arm, CFG, DEVICE, resume=True)
    res.update(
        cfg={k: v for k, v in CFG.items()},
        data=data.cpu(),
        note=NOTE,
        gmm_gate=GATE,
        D_minus=D_MINUS,
        scale=SCALE,
        sigma_max=CFG['sigma_max'],
        lam_emp={k: v.cpu() for k, v in lam_emp.items()},
        lam_pop=lam_pop.cpu(),
        c_values=C_VALUES,
        null_rel_threshold=NULL_REL_THRESHOLD,
    )
    torch.save(res, p)
    print(f"  done in {res['minutes']:.1f} min -> {p}", flush=True)

# Summary
print('\nAll arms on disk:')
for arm in ARMS:
    p = arm_result_path(arm['variant'], arm['c'])
    status = 'EXISTS' if os.path.exists(p) else 'MISSING'
    print(f'  {arm["label"]:>25s}  {status}  {p}')

## Results

### Load all completed arms

In [ ]:
def load_all_arms():
    '''Load every completed arm from disk.'''
    out = {}
    for arm in ARMS:
        p = arm_result_path(arm['variant'], arm['c'])
        if os.path.exists(p):
            out[(arm['variant'], arm['c'])] = torch.load(p, map_location='cpu', weights_only=False)
    return out

all_runs = load_all_arms()
print(f'Loaded {len(all_runs)} arms from disk:')
for (v, c), r in sorted(all_runs.items(), key=lambda x: (x[0][0], x[0][1])):
    last = r['eval_log'][-1] if r['eval_log'] else {}
    print(f'  {v:>15s} c={c:<8g} epochs={last.get("epoch", "?"):>6} '
          f'frac={last.get("fraction", float("nan")):.2f} '
          f'coarse={last.get("coarse_score", float("nan")):.4f} '
          f'fine={last.get("fine_score", float("nan")):.4f}')

### Gate check

The gate arm ($c=0$) must match `arm_rescaled_c16_result.pt` from `baptista_config_matern_n2.ipynb`:
fraction ~ 0.99 by epoch 29k.

In [ ]:
gate_run = all_runs.get(('gate', 0.0))
if gate_run:
    # Check against the existing result file
    ref_path = os.path.join(results_dir, 'baptista_matern_n2', 'arm_rescaled_c16_result.pt')
    if os.path.exists(ref_path):
        ref = torch.load(ref_path, map_location='cpu', weights_only=False)
        ref_frac = [e['fraction'] for e in ref['eval_log']]
        ref_ep = [e['epoch'] for e in ref['eval_log']]
        gate_frac = [e['fraction'] for e in gate_run['eval_log']]
        gate_ep = [e['epoch'] for e in gate_run['eval_log']]
        print('Gate arm vs arm_rescaled_c16_result.pt:')
        print(f'  reference: {len(ref_frac)} evals, peak fraction {max(ref_frac):.2f}')
        print(f'  gate arm:  {len(gate_frac)} evals, peak fraction {max(gate_frac):.2f}')
        # Check that gate reaches similar memorization level
        if max(gate_frac) >= 0.9:
            print('  GATE CHECK PASSED: gate arm memorizes as expected.')
        else:
            print('  GATE CHECK WARNING: gate arm has not reached full memorization yet.')
    else:
        print(f'  reference file not found: {ref_path}')
        gate_frac = [e['fraction'] for e in gate_run['eval_log']]
        print(f'  gate arm: {len(gate_frac)} evals, peak fraction {max(gate_frac):.2f}')
else:
    print('Gate arm not yet trained.')

### Headline: coarse and fine band scores vs c, all variants

In [ ]:
if len(all_runs) >= 2:
    cov_emp_variants = [f'cov_emp_n{n}' for n in ESTIMATION_NS]
    cov_emp_colors = {f'cov_emp_n{n}': c for n, c in
                      zip(ESTIMATION_NS, ['tab:green', 'tab:cyan', 'tab:purple', 'tab:brown'])}
    styles = {'isotropic': dict(color='tab:gray', marker='o', lw=1.8)}
    for v in cov_emp_variants:
        styles[v] = dict(color=cov_emp_colors[v], marker='^', lw=1.4)
    styles['cov_population'] = dict(color='tab:olive', marker='v', lw=1.8, ls='--')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, band in zip(axes, ['coarse', 'fine']):
        for variant, sty in styles.items():
            cs, ys = [], []
            for c in C_VALUES:
                key = (variant, c)
                if key in all_runs:
                    last = all_runs[key]['eval_log'][-1]
                    cs.append(c)
                    ys.append(last[f'{band}_score'])
            if cs:
                ax.plot(cs, ys, label=variant, **sty)

        gate_key = ('gate', 0.0)
        if gate_key in all_runs:
            gate_last = all_runs[gate_key]['eval_log'][-1]
            ax.axhline(gate_last[f'{band}_score'], color='tab:red', lw=1.0, ls='--',
                       label='gate (unregularized)')
        ax.axhline(1.0, color='gray', lw=0.8, ls='--')
        ax.set_xscale('log')
        ax.set_xlabel('c')
        ax.set_title(f'{band} band')
    axes[0].set_ylabel('band score (mean ratio; <1 = memorized)')
    axes[1].legend(fontsize=7)
    fig.suptitle('SongUNet covariance Tikhonov: band scores vs c (final epoch)', y=1.04)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'songunet_cov_tikhonov_band_scores.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough arms completed for the headline plot.')


### Training curves: fraction and band scores over epochs

In [ ]:
if len(all_runs) >= 2:
    cov_emp_colors = {f'cov_emp_n{n}': c for n, c in
                      zip(ESTIMATION_NS, ['tab:green', 'tab:cyan', 'tab:purple', 'tab:brown'])}
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

    for (variant, c_val), r in sorted(all_runs.items(), key=lambda x: (x[0][0], x[0][1])):
        log_entries = r['eval_log']
        if not log_entries:
            continue
        ep = [e['epoch'] for e in log_entries]
        label = f'{variant} c={c_val:g}'
        if variant == 'gate':
            color, ls = 'tab:red', '-'
        elif variant == 'isotropic':
            color, ls = 'tab:gray', '-'
        elif variant.startswith('cov_emp'):
            color, ls = cov_emp_colors.get(variant, 'tab:green'), '-'
        else:
            color, ls = 'tab:olive', '--'

        axes[0].plot(ep, [e['fraction'] for e in log_entries],
                     color=color, ls=ls, lw=1.2, label=label)
        axes[1].plot(ep, [e['coarse_score'] for e in log_entries],
                     color=color, ls=ls, lw=1.2, label=label)
        axes[2].plot(ep, [e['fine_score'] for e in log_entries],
                     color=color, ls=ls, lw=1.2, label=label)

    axes[0].set_ylabel('collapse fraction')
    axes[0].set_ylim(-0.02, 1.02)
    axes[0].set_title('fraction < 0.3')
    axes[1].set_ylabel('coarse band score')
    axes[1].set_title('coarse (0.5-4) -- <1 = memorized')
    axes[2].set_ylabel('fine band score')
    axes[2].set_title('fine (18-32) -- <1 = memorized')
    for ax in axes:
        ax.set_xlabel('Epochs')
        ax.legend(fontsize=6)
        ax.axhline(1.0, color='gray', lw=0.5, ls=':')
    fig.suptitle('SongUNet covariance Tikhonov: training curves', y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'songunet_cov_tikhonov_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()


### Per-wavenumber ratio at final epoch

In [ ]:
if len(all_runs) >= 2:
    kc = ctx.k_centers.cpu().numpy()
    variants_to_show = ['isotropic', 'cov_emp_n2', 'cov_population']
    n_panels = len(variants_to_show)
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 4.2), sharey=True)
    cmap_c = plt.cm.viridis

    for ax, variant in zip(axes, variants_to_show):
        gate_key = ('gate', 0.0)
        if gate_key in all_runs:
            gate_mr = all_runs[gate_key]['eval_log'][-1]['mean_ratio'].numpy()
            ax.plot(kc, gate_mr, color='tab:red', lw=1.8, label='gate (c=0)')

        for j, c_val in enumerate(C_VALUES):
            key = (variant, c_val)
            if key in all_runs:
                mr = all_runs[key]['eval_log'][-1]['mean_ratio'].numpy()
                ax.plot(kc, mr, color=cmap_c(j / max(len(C_VALUES)-1, 1)),
                        lw=1.5, label=f'c={c_val:g}')

        for bname, color in [('coarse', 'tab:blue'), ('fine', 'tab:red')]:
            lo, hi = bands[bname]
            ax.axvspan(lo, hi, color=color, alpha=0.08)
        ax.axhline(1.0, color='gray', lw=0.8, ls='--')
        ax.set_xlim(0, 45)
        ax.set_xlabel('wavenumber k')
        ax.set_title(variant)
    axes[0].set_ylabel('mean ratio (err_NN / err_random)')
    axes[0].legend(fontsize=7)
    fig.suptitle('Per-wavenumber memorization ratio at final epoch', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'songunet_cov_tikhonov_per_k.png'),
                dpi=150, bbox_inches='tight')
    plt.show()


### Summary table

In [ ]:
print(f'{"arm":>25s} {"c":>8s}  {"fraction":>8s}  {"coarse":>8s}  {"mid1":>8s}  '
      f'{"mid2":>8s}  {"fine":>8s}')
print('-' * 85)
for (variant, c_val), r in sorted(all_runs.items(), key=lambda x: (x[0][0], x[0][1])):
    if not r['eval_log']:
        continue
    last = r['eval_log'][-1]
    print(f'{variant:>25s} {c_val:>8g}  {last["fraction"]:>8.3f}  '
          f'{last["coarse_score"]:>8.4f}  {last.get("mid1_score", float("nan")):>8.4f}  '
          f'{last.get("mid2_score", float("nan")):>8.4f}  {last["fine_score"]:>8.4f}')

### Samples vs training time (gate arm)

In [ ]:
gate_run = all_runs.get(('gate', 0.0))
if gate_run and gate_run['eval_log']:
    log_rows = gate_run['eval_log']
    want = [2000, 5000, 10000, 20000, 30000, 50000]
    avail = [e['epoch'] for e in log_rows]
    picks, seen = [], set()
    for w in want:
        j = int(np.argmin([abs(a - w) for a in avail]))
        if j not in seen:
            seen.add(j); picks.append(j)

    n_show = min(6, log_rows[0]['samples'].shape[0])
    vmax = float(data.abs().max())
    fig, axes = plt.subplots(len(picks) + 1, n_show, squeeze=False,
                             figsize=(1.1 * n_show, 1.15 * (len(picks) + 1)))
    for row, j in enumerate(picks):
        s = log_rows[j]['samples']
        for k in range(n_show):
            axes[row][k].imshow(s[k, 0], vmin=-vmax, vmax=vmax, cmap='RdBu_r')
            axes[row][k].set_xticks([]); axes[row][k].set_yticks([])
        axes[row][0].set_ylabel(f"{log_rows[j]['epoch']//1000}k\n"
                                f"{log_rows[j].get('nn_rel_median', 0):.3f}",
                                fontsize=7, rotation=0, ha='right', va='center')
    for k in range(n_show):
        ax = axes[-1][k]
        ax.set_xticks([]); ax.set_yticks([])
        if k < data.shape[0]:
            ax.imshow(data[k, 0], vmin=-vmax, vmax=vmax, cmap='RdBu_r')
        else:
            ax.axis('off')
    axes[-1][0].set_ylabel('train', fontsize=7, rotation=0, ha='right', va='center')
    fig.suptitle(f'Gate arm samples vs training time (model_channels={MODEL_CHANNELS}, '
                 f'{gate_run["n_params"]:,} params)', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'songunet_cov_tikhonov_gate_samples.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Gate arm not available for sample visualization.')

## Notes

* **What to look for:** In the closed-form GMM, covariance weighting de-memorizes the fine
  band at $c$ ~100x smaller than isotropic while the coarse band stays memorized. If the
  SongUNet curves reproduce this, the scale-selective regularizer transfers to the learned
  setting at $n_{\text{train}}=2$ -- the paper's central claim.
* **If they don't track:** the network's inductive bias may dominate at this capacity.
  Run the gate arm first to confirm unregularized memorization, then interpret.
* **Reduction mismatch.** Baptista's training loop uses `loss.sum() / B` (sum over pixels),
  while `src/edm.py:tikhonov_penalty()` returns a per-sample mean over pixels. The notebook-local
  penalty functions use sum reduction to match. The effective constant is the same as $c$ in the
  closed-form expressions.
* **sigma_data.** Held at 0.5 for consistency with the source notebook, even though the rescaled
  data has std ~ 0.10. A sigma_data=0.1 arm would be a useful check.
* **Empirical vs population spectrum.** Both are tested to check robustness of the covariance
  estimate from only 2 training fields. The uncentered periodogram achieves ~10% mean error
  vs population on the rescaled data.